# Development of an Annual Anthropogenic Pressure Product for Eastern Morocco
- Created on 9/11/2026

## 1. Purpose

This workflow will develop a locally adapted, spatially detailed anthropogenic
pressure dataset for Eastern Morocco. **The product will complement two existing
families of ecological covariates**:

1. annual live vegetation cover predictions, representing dynamic resource availability;
2. annual habitat maps, representing habitat identity, extent, and spatial configuration.

The anthropogenic pressure product will represent human accessibility, infrastructure, settlement, land transformation, and observed activity. It will support both Houbara Bustard resource selection modelling and analysis of the drivers of habitat loss, conversion, persistence, and fragmentation.

The principal temporal period will be **2008–2024**, corresponding to the available field anthropization records. The overlapping period with the annual habitat maps will be used for habitat-change analyses.

---

## 2. Conceptual distinction among products

| Product | Ecological meaning | Data form |
|---|---|---|
| Live vegetation cover | Temporally changing trophic or vegetation resource availability | Continuous raster |
| Habitat map | Ecological or land-cover state present in a given year | Categorical raster |
| Habitat/LULC change | Transition between habitat states across years | Transition or change raster |
| Anthropogenic pressure | Potential or observed human influence on a location | Continuous component layers and indices |

Anthropogenic pressure is therefore treated as a potential driver or exposure
variable rather than as another habitat classification. A habitat can remain
unchanged while pressure increases because of new access, buildings, settlements,
wells, cultivation, or human activity.

---

## 3. Initial data sources

| Data source | Main information | Proposed temporal treatment |
|---|---|---|
| Field anthropization records, 2008–2024 | Human activity, camps, livestock, infrastructure, disturbance, or other recorded pressures | Annual, seasonal, event-based, or persistent depending on feature type |
| `hotosm_mar_roads_lines_shp` | Roads, tracks, accessibility, and road classes | Persistent or stepwise; historical status requires verification |
| `hotosm_mar_populated_places_points/polygons_shp` | Cities, towns, villages, hamlets, and isolated dwellings | Settlement reference; stepwise if expansion dates can be identified |
| `hotosm_mar_railways_lines_shp` | Railway lines and stations | Primarily persistent infrastructure |
| HOTOSM/Open Buildings polygons | Recent building footprints and building density | Recent reference layer; not automatically back-cast to earlier years |
| Annual habitat maps | Cultivated/fallow and built-up extent | Annual land-transformation information; used carefully to avoid circularity |
| Historical and recent satellite imagery | Verification of infrastructure and settlement timing | Used to identify first-observed year and validate changes |
| Global Human Footprint, 2000–2018 | Coarse regional comparison | Benchmark and validation only, not the primary input |

Because OpenStreetMap/HOTOSM layers describe the mapped database at the time of
export rather than the true construction year of each object, they will be
treated as reference snapshots unless historical existence is independently
verified.

---

## 4. Temporal design

The final pressure product will have annual outputs, but individual components
will be assigned to different temporal classes.

### Persistent or slowly changing features

Examples include long-established major roads, railways, permanent wells, and
established settlements. These features remain active across years after their
verified start date.

### Stepwise-changing features

Examples include new buildings, roads, settlement expansion, and permanent
infrastructure. Their values change only when a new feature is first observed.

### Annually dynamic features

Examples include annual cultivation, annual built-up expansion, and field
anthropization observations that represent conditions during a particular year.

### Seasonal or event-based features

Examples include temporary camps, people, vehicles, or livestock observations.
These records will not be treated as permanent unless repeated observations or
other evidence support persistence.

Each feature should ideally contain:

- `feature_id`
- `pressure_type`
- `pressure_subtype`
- `source`
- `observation_date`
- `start_year`
- `end_year`
- `temporal_class`
- `intensity`
- `confidence`
- `survey_effort`
- `geometry`

Absence of a field record will not automatically be interpreted as absence of
human pressure, particularly where survey effort varied among locations or years.

---

## 5. Processing framework

### Step 1: Data inventory and harmonization

- inspect geometry types, attributes, spatial coverage, and temporal information;
- standardize feature names and anthropization categories;
- repair invalid geometries;
- transform all data to a common projected coordinate reference system;
- clip the inputs to the Eastern Morocco study region;
- identify duplicate or overlapping records;
- assign temporal and confidence classes.

### Step 2: Develop component pressure layers

Candidate component layers include:

- distance to major roads;
- distance to secondary roads;
- distance to unpaved roads or tracks;
- road density within multiple neighborhood sizes;
- distance to populated places;
- distance to buildings;
- building count and building-footprint density;
- distance to railways;
- distance to wells or water-access infrastructure;
- field-observed human-activity intensity;
- cultivated-area fraction;
- built-up-area fraction;
- distance to the existing cultivation or settlement edge.

Distance effects may be represented using continuous decay functions, while
density and fractional-cover variables will be calculated using ecologically
relevant neighborhood sizes.

### Step 3: Organize components into pressure domains

Four initial domains are proposed:

1. **Access and transportation**
   - roads, tracks, and railways;

2. **Settlements and buildings**
   - populated places, individual buildings, and building density;

3. **Land transformation**
   - cultivated/fallow land, built-up land, mining, quarrying, or permanent
     clearings;

4. **Observed human activity**
   - field-recorded camps, people, livestock, vehicles, and localized disturbance.

### Step 4: Produce annual pressure surfaces

For each year \(t\), the domain layers will be combined as:

\[
HPI_{x,t} =
w_I I_{x,t} +
w_S S_{x,t} +
w_L L_{x,t} +
w_A A_{x,t},
\]

where \(I\), \(S\), \(L\), and \(A\) represent the four pressure domains.

Equal domain weights will provide the initial transparent formulation. Alternative
weights may later be evaluated using field observations, expert knowledge, model
performance, and sensitivity analysis.

All years will be normalized using fixed thresholds derived from the complete
study period. Each year will not be independently rescaled, because independent
annual scaling would prevent valid temporal comparison.

### Step 5: Validation and sensitivity analysis

The annual products will be evaluated using:

- field anthropization records withheld from development;
- spatially blocked validation;
- temporal withholding of selected years;
- visual interpretation of high-resolution imagery;
- comparison with the coarse global Human Footprint after aggregation;
- sensitivity to neighborhood sizes, decay distances, component weights, and
  uncertain infrastructure dates.

A confidence or data-quality surface will be produced in addition to the
anthropogenic pressure score.

---

## 6. Expected outputs

The workflow will generate:

- annual component rasters for 2008–2024;
- annual pressure-domain rasters;
- an optional composite Human Pressure Index;
- distance and density covariates for ecological modelling;
- a temporal-confidence or data-quality raster;
- summaries of pressure by habitat, region, and year;
- maps of persistent pressure, recent expansion, and pressure trends;
- metadata documenting source year, temporal treatment, and uncertainty.

Provisional output names may include:

- `EM_Road_Access_YYYY.tif`
- `EM_Settlement_Pressure_YYYY.tif`
- `EM_Land_Transformation_YYYY.tif`
- `EM_Observed_Activity_YYYY.tif`
- `EM_Human_Pressure_Index_YYYY.tif`
- `EM_Human_Pressure_Confidence_YYYY.tif`

A 30-m grid is a defensible starting resolution for the complete 2008–2024
series. A higher-resolution enhancement may subsequently be produced for the
Sentinel-2 period, provided that the apparent spatial detail is supported by the
underlying temporal data.

---

## 7. Intended applications

### Resource selection functions

The pressure layer will be matched to the year, season, or observation period of
the Houbara locations. Component or domain variables will generally be preferred
over a single composite index because roads, settlements, cultivation, and field
activity may influence resource selection through different mechanisms.

The composite index and all of its component variables will not be included in
the same model because this would duplicate information and increase
collinearity.

### Habitat-change analysis

Anthropogenic pressure from year \(t-1\) will be used to explain habitat change
between \(t-1\) and \(t\). Components that directly define the response, such as
same-year built-up or cultivated cover, will be excluded from the explanatory
index when modelling conversion to those land-cover classes.

---

## 8. Expected contribution and limitations

The product is expected to provide substantially greater local ecological detail
than existing global 1-km Human Footprint datasets, particularly for roads,
tracks, settlements, buildings, pastoral activity, and field-observed
anthropization.

However, it will not represent a perfectly observed historical record. Important
limitations include:

- incomplete or spatially uneven OpenStreetMap coverage;
- uncertain construction dates for contemporary infrastructure;
- variation in field survey effort;
- incomplete detection of temporary human activity;
- possible overlap between land-cover variables and anthropogenic-pressure
  variables;
- variable temporal reliability among components.

These limitations will be addressed through explicit temporal classes,
source-specific confidence scores, validation, and sensitivity analysis.

# Eastern Morocco anthropogenic-pressure workflow

## Starter notebook: inventory, harmonization, and pilot pressure layers

This notebook establishes the first reproducible code base for a locally adapted anthropogenic-pressure product for Eastern Morocco. It is designed to complement:

1. annual/seasonal live vegetation cover predictions;
2. annual habitat classifications;
3. Houbara Bustard resource-selection and habitat-change analyses.

### Temporal rule used in this notebook

- HOTOSM roads, settlements, railways, and buildings are treated first as a **current reference snapshot**.
- They are **not automatically back-cast to 2008**.
- Field anthropization records are assigned annual, event-based, semi-permanent, or persistent temporal behavior.
- Annual habitat-derived land-transformation layers can be added after the yearly habitat maps are available.

### First outputs

- standardized and inventoried input layers;
- a common 30-m processing grid in UTM Zone 30N;
- current-reference distances and neighborhood densities for roads, tracks, settlements, railways, and buildings;
- annual pilot layers from dated field anthropization records for 2008, 2016, and 2024;
- metadata and inventory tables;
- functions ready to incorporate cultivated/fallow and built-up fractions from annual habitat maps.

The composite Human Pressure Index is intentionally **not finalized in this first notebook**. Component layers should be inspected and validated before weights and decay distances are selected.


## 0. Environment

Recommended conda installation, if any packages are missing:

```bash
conda install -c conda-forge geopandas pyogrio pyarrow rasterio scipy shapely pandas numpy matplotlib jupyter
```

The notebook uses projected metric coordinates because all distances, buffers, and neighborhood sizes are expressed in meters.


In [1]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Iterable, Mapping, Sequence
import json
import math
import re
import unicodedata
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from IPython.display import display

import rasterio
from affine import Affine
from rasterio import features
from rasterio.crs import CRS
from rasterio.enums import MergeAlg, Resampling
from rasterio.transform import from_origin
from rasterio.warp import reproject

from scipy.ndimage import distance_transform_edt
from scipy.signal import fftconvolve

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

print("GeoPandas:", gpd.__version__)
print("Rasterio:", rasterio.__version__)


GeoPandas: 1.1.4
Rasterio: 1.5.1


## 1. User configuration

Edit only this section first. A path may point either to a vector file or to a directory containing one shapefile. The helper function below will resolve a single `.shp` inside an extracted HOTOSM folder.

`EPSG:32630` is appropriate for most of Eastern Morocco west of 0° longitude. Verify that the complete study AOI lies in UTM Zone 30N before processing.


In [ ]:
# -----------------------------------------------------------------------------
# PROJECT PATHS: EDIT THESE
# -----------------------------------------------------------------------------
PROJECT_ROOT = Path(
    r"C:\Users\PangY\OneDrive - Smithsonian Institution\Bustard\01_Data"
)

# Replace with the actual Eastern Morocco study-area polygon.
AOI_PATH = PROJECT_ROOT / "Morocco two study areas" / "Eastern_Morocco_Working_Area.shp"
AOI_LAYER = None  # Set a GeoPackage layer name only when needed.

# Replace with the actual field anthropization file for 2008-2024.
ANTHRO_PATH = (
    PROJECT_ROOT
    / "Reneco"
    / "GIS Data"
    / "Anthropization_2015_2025"
    / "Anthropization_2015_2025.shp"
)
ANTHRO_LAYER = None

# Point these paths to the extracted HOTOSM folders or directly to .shp files.
HOTOSM_ROOT = PROJECT_ROOT / "Habitat mapping" / "build-up"
ROADS_PATH = HOTOSM_ROOT / "hotosm_mar_roads_lines_shp" / "hotosm_mar_roads_lines_shp_EM.shp
BUILDINGS_PATH = HOTOSM_ROOT / "hotosm_mar_buildings_polygons_shp" / "hotosm_mar_buildings_polygons_shp.shp"
PLACES_POLYGONS_PATH = HOTOSM_ROOT / "hotosm_mar_populated_places_polygons_shp" / "hotosm_mar_populated_places_polygons_shp.shp"
RAILWAYS_PATH = HOTOSM_ROOT / "hotosm_mar_railways_lines_shp" / "hotosm_mar_railways_lines_shp"

# Annual habitat maps will be incorporated later. Edit the naming pattern when ready.
HABITAT_RASTER_TEMPLATE = (
    PROJECT_ROOT
    / "Habitat_mapping"
    / "Annual_maps"
    / "EM_Habitat_{year}.tif"
)

OUTPUT_ROOT = PROJECT_ROOT / "Human_Footprint" / "EM_Anthropogenic_Pressure"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# -----------------------------------------------------------------------------
# SPATIAL AND TEMPORAL SETTINGS
# -----------------------------------------------------------------------------
TARGET_CRS = "EPSG:32630"
RESOLUTION_M = 30
PILOT_YEARS = [2008, 2016, 2024]
FULL_YEAR_RANGE = range(2008, 2025)

# The processing grid extends beyond the AOI so that features just outside the
# boundary can influence distance/density values inside the AOI.
MAX_CONTEXT_M = 5_000
DENSITY_RADII_M = [500, 1_000, 5_000]

NODATA_FLOAT = -9999.0
OVERWRITE = False

# Edit these codes after confirming the final habitat legend/codebook.
HABITAT_CODES = {
    "cultivated_fallow": [9],
    "built_up": [11],
}

print("Output directory:", OUTPUT_ROOT)


## 2. General vector utilities

These functions:

- resolve a vector file from a path or extracted shapefile folder;
- normalize field names;
- repair invalid geometries;
- reproject all layers to the target CRS;
- clip source layers to the buffered processing extent;
- summarize schema, geometry, and coverage.


In [ ]:
def normalize_text(value: object) -> str:
    """Normalize free text for robust category matching."""
    if value is None or pd.isna(value):
        return ""
    text = unicodedata.normalize("NFKD", str(value))
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = text.strip().lower()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    return text.strip("_")


def make_unique_names(names: Iterable[object]) -> list[str]:
    """Create normalized, unique field names while preserving geometry."""
    counts: dict[str, int] = {}
    result: list[str] = []
    for name in names:
        base = "geometry" if str(name).lower() == "geometry" else normalize_text(name)
        base = base or "field"
        counts[base] = counts.get(base, 0) + 1
        result.append(base if counts[base] == 1 else f"{base}_{counts[base]}")
    return result


def resolve_vector_path(path: str | Path) -> Path:
    """Resolve a vector file, including a folder containing exactly one .shp."""
    path = Path(path)
    if path.is_file():
        return path
    if path.is_dir():
        shapefiles = sorted(path.glob("*.shp"))
        if len(shapefiles) == 1:
            return shapefiles[0]
        if len(shapefiles) == 0:
            raise FileNotFoundError(f"No .shp file found in directory: {path}")
        raise ValueError(
            f"More than one .shp file found in {path}. Point directly to the desired file:\n"
            + "\n".join(str(p) for p in shapefiles)
        )
    raise FileNotFoundError(f"Vector path does not exist: {path}")


def union_geometry(gdf: gpd.GeoDataFrame):
    """Return one unioned geometry with compatibility across GeoPandas versions."""
    if hasattr(gdf.geometry, "union_all"):
        return gdf.geometry.union_all()
    return gdf.geometry.unary_union


def _read_file_fast(path: Path, layer: str | None = None) -> gpd.GeoDataFrame:
    """Prefer Pyogrio/Arrow, with a conservative fallback."""
    kwargs = {"layer": layer} if layer else {}
    try:
        return gpd.read_file(path, engine="pyogrio", use_arrow=True, **kwargs)
    except Exception as exc:
        warnings.warn(f"Fast read failed for {path.name}; falling back to default engine. {exc}")
        return gpd.read_file(path, **kwargs)


def load_vector(
    path: str | Path,
    *,
    name: str,
    target_crs: str | CRS,
    layer: str | None = None,
    clip_geometry=None,
    required: bool = True,
) -> gpd.GeoDataFrame | None:
    """Load, clean, project, and optionally clip a vector layer."""
    try:
        resolved = resolve_vector_path(path)
    except (FileNotFoundError, ValueError):
        if required:
            raise
        warnings.warn(f"Optional layer not found and will be skipped: {name} -> {path}")
        return None

    gdf = _read_file_fast(resolved, layer=layer)
    gdf.columns = make_unique_names(gdf.columns)

    if "geometry" not in gdf.columns:
        raise ValueError(f"{name} has no geometry column: {resolved}")
    if gdf.crs is None:
        raise ValueError(
            f"{name} has no CRS. Assign its correct source CRS before continuing: {resolved}"
        )

    gdf = gdf.loc[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()
    if gdf.empty:
        warnings.warn(f"{name} contains no non-empty geometries after loading.")
        return gdf.to_crs(target_crs)

    # GeoSeries.make_valid is vectorized in current GeoPandas/Shapely versions.
    try:
        gdf.geometry = gdf.geometry.make_valid()
    except Exception as exc:
        warnings.warn(f"Geometry repair was not completed for {name}: {exc}")

    gdf = gdf.to_crs(target_crs)

    if clip_geometry is not None and not gdf.empty:
        mask = gpd.GeoDataFrame(geometry=[clip_geometry], crs=target_crs)
        gdf = gpd.clip(gdf, mask, keep_geom_type=False)
        gdf = gdf.loc[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()

    # Separate multipart records so later rasterization is more predictable.
    if not gdf.empty:
        gdf = gdf.explode(index_parts=False, ignore_index=True)

    gdf["source_layer"] = name
    gdf["source_path"] = str(resolved)
    return gdf.reset_index(drop=True)


def layer_inventory(name: str, gdf: gpd.GeoDataFrame | None) -> dict[str, object]:
    if gdf is None:
        return {
            "layer": name,
            "status": "not_loaded",
            "n_features": 0,
            "geometry_types": "",
            "crs": "",
            "columns": "",
        }
    bounds = gdf.total_bounds if not gdf.empty else [np.nan] * 4
    return {
        "layer": name,
        "status": "loaded",
        "n_features": int(len(gdf)),
        "geometry_types": ", ".join(sorted(gdf.geometry.geom_type.unique())) if len(gdf) else "",
        "crs": str(gdf.crs),
        "minx": bounds[0],
        "miny": bounds[1],
        "maxx": bounds[2],
        "maxy": bounds[3],
        "columns": ", ".join(gdf.columns),
    }


def first_existing_column(
    gdf: gpd.GeoDataFrame,
    candidates: Sequence[str],
    explicit: str | None = None,
) -> str | None:
    """Resolve an explicitly supplied field or the first candidate present."""
    if explicit:
        explicit_norm = normalize_text(explicit)
        if explicit_norm not in gdf.columns:
            raise KeyError(
                f"Configured field '{explicit}' was not found. Available fields: {list(gdf.columns)}"
            )
        return explicit_norm
    for candidate in candidates:
        candidate_norm = normalize_text(candidate)
        if candidate_norm in gdf.columns:
            return candidate_norm
    return None


## 3. Load the AOI and source layers

The AOI itself is required. HOTOSM point settlements and buildings are optional in the first run; the notebook continues when an optional layer is unavailable.


In [ ]:
# Load and dissolve the Eastern Morocco AOI.
aoi = load_vector(
    AOI_PATH,
    name="eastern_morocco_aoi",
    target_crs=TARGET_CRS,
    layer=AOI_LAYER,
    required=True,
)
assert aoi is not None and not aoi.empty

aoi_geometry = union_geometry(aoi)
processing_geometry = aoi_geometry.buffer(MAX_CONTEXT_M)

# Load all vector sources within the buffered context area.
roads = load_vector(
    ROADS_PATH,
    name="hotosm_roads",
    target_crs=TARGET_CRS,
    clip_geometry=processing_geometry,
)
places_polygons = load_vector(
    PLACES_POLYGONS_PATH,
    name="hotosm_populated_places_polygons",
    target_crs=TARGET_CRS,
    clip_geometry=processing_geometry,
)
places_points = load_vector(
    PLACES_POINTS_PATH,
    name="hotosm_populated_places_points",
    target_crs=TARGET_CRS,
    clip_geometry=processing_geometry,
    required=False,
)
railways = load_vector(
    RAILWAYS_PATH,
    name="hotosm_railways",
    target_crs=TARGET_CRS,
    clip_geometry=processing_geometry,
)
buildings = load_vector(
    BUILDINGS_PATH,
    name="hotosm_buildings",
    target_crs=TARGET_CRS,
    clip_geometry=processing_geometry,
    required=False,
)
anthro_raw = load_vector(
    ANTHRO_PATH,
    name="field_anthropization_2008_2024",
    target_crs=TARGET_CRS,
    layer=ANTHRO_LAYER,
    clip_geometry=processing_geometry,
    required=False,
)

layers = {
    "aoi": aoi,
    "roads": roads,
    "places_polygons": places_polygons,
    "places_points": places_points,
    "railways": railways,
    "buildings": buildings,
    "anthropization": anthro_raw,
}

inventory = pd.DataFrame(layer_inventory(name, gdf) for name, gdf in layers.items())
display(inventory)

inventory_path = OUTPUT_ROOT / "input_layer_inventory.csv"
inventory.to_csv(inventory_path, index=False)
print("Saved:", inventory_path)


### Inspect source attributes before classification

Run this cell and inspect the values carefully. HOTOSM field names are usually recognizable (`highway`, `place`, `railway`, `building`), but the code below also searches common alternatives.


In [ ]:
for name, gdf in layers.items():
    print("\n" + "=" * 90)
    print(name.upper())
    print("=" * 90)
    if gdf is None:
        print("Not available")
        continue
    print("Features:", len(gdf))
    print("Geometry:", gdf.geometry.geom_type.value_counts(dropna=False).to_dict())
    print("Fields:", list(gdf.columns))
    display(gdf.drop(columns="geometry", errors="ignore").head(5))


## 4. Standardize HOTOSM thematic classes

The classes below are initial ecological groupings, not final weights. Keep the original HOTOSM attributes in the standardized GeoDataFrames for later review.


In [ ]:
ROAD_GROUPS = {
    "major": {"motorway", "motorway_link", "trunk", "trunk_link", "primary", "primary_link"},
    "secondary": {"secondary", "secondary_link", "tertiary", "tertiary_link"},
    "local": {
        "residential", "unclassified", "service", "living_street", "road",
        "pedestrian",
    },
    "track": {"track", "path", "bridleway"},
}

PLACE_GROUPS = {
    "urban_center": {"city", "town"},
    "village": {"village"},
    "small_settlement": {
        "hamlet", "isolated_dwelling", "neighbourhood", "quarter", "locality",
    },
}

ACTIVE_RAIL_TYPES = {
    "rail", "light_rail", "narrow_gauge", "tram", "subway", "monorail",
}

INVALID_BUILDING_TAGS = {
    "no", "demolished", "destroyed", "razed", "proposed", "construction",
}


def map_category(value: object, groups: Mapping[str, set[str]], default: str = "other") -> str:
    text = normalize_text(value)
    # Handle semicolon-separated OpenStreetMap values.
    tokens = set(text.split("_")) | {normalize_text(v) for v in str(value).split(";")}
    for group, accepted in groups.items():
        if text in accepted or tokens.intersection(accepted):
            return group
    return default


def standardize_roads(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    result = gdf.copy()
    field = first_existing_column(result, ["highway", "fclass", "road_type", "type"])
    if field is None:
        raise KeyError(f"No road-class field found. Fields: {list(result.columns)}")
    result["road_type_raw"] = result[field].astype("string")
    result["road_type"] = result["road_type_raw"].map(normalize_text)
    result["road_group"] = result["road_type_raw"].map(lambda x: map_category(x, ROAD_GROUPS))
    return result


def standardize_places(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    result = gdf.copy()
    field = first_existing_column(result, ["place", "fclass", "place_type", "type"])
    if field is None:
        result["place_type_raw"] = "unknown"
        result["place_type"] = "unknown"
        result["place_group"] = "other"
        return result
    result["place_type_raw"] = result[field].astype("string")
    result["place_type"] = result["place_type_raw"].map(normalize_text)
    result["place_group"] = result["place_type_raw"].map(lambda x: map_category(x, PLACE_GROUPS))
    return result


def standardize_railways(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    result = gdf.copy()
    field = first_existing_column(result, ["railway", "fclass", "rail_type", "type"])
    if field is None:
        result["rail_type_raw"] = "unknown"
        result["rail_type"] = "unknown"
        result["rail_active_candidate"] = True
        return result
    result["rail_type_raw"] = result[field].astype("string")
    result["rail_type"] = result["rail_type_raw"].map(normalize_text)
    result["rail_active_candidate"] = result["rail_type"].isin(ACTIVE_RAIL_TYPES)
    return result


def standardize_buildings(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    result = gdf.copy()
    field = first_existing_column(result, ["building", "fclass", "building_type", "type"])
    if field is None:
        result["building_type_raw"] = "yes"
        result["building_type"] = "yes"
        return result
    result["building_type_raw"] = result[field].astype("string")
    result["building_type"] = result["building_type_raw"].map(normalize_text)
    result = result.loc[~result["building_type"].isin(INVALID_BUILDING_TAGS)].copy()
    return result


roads_std = standardize_roads(roads)
places_polygons_std = standardize_places(places_polygons)
places_points_std = standardize_places(places_points) if places_points is not None else None
railways_std = standardize_railways(railways)
buildings_std = standardize_buildings(buildings) if buildings is not None else None

print("Road groups:")
display(roads_std["road_group"].value_counts(dropna=False).rename_axis("road_group").to_frame("n"))
print("\nPopulated-place groups (polygons):")
display(places_polygons_std["place_group"].value_counts(dropna=False).rename_axis("place_group").to_frame("n"))
print("\nRailway types:")
display(railways_std["rail_type"].value_counts(dropna=False).head(20).rename_axis("rail_type").to_frame("n"))
if buildings_std is not None:
    print("\nBuilding types:")
    display(buildings_std["building_type"].value_counts(dropna=False).head(20).rename_axis("building_type").to_frame("n"))


## 5. Build the common processing grid

The grid covers the AOI plus a 5-km context buffer. Outputs are masked to the original AOI, but roads or settlements just outside the AOI can still influence locations close to the boundary.


In [ ]:
@dataclass(frozen=True)
class GridSpec:
    crs: CRS
    transform: Affine
    width: int
    height: int
    resolution: float
    bounds: tuple[float, float, float, float]


def aligned_grid(geometry, crs: str | CRS, resolution: float) -> GridSpec:
    minx, miny, maxx, maxy = geometry.bounds
    left = math.floor(minx / resolution) * resolution
    bottom = math.floor(miny / resolution) * resolution
    right = math.ceil(maxx / resolution) * resolution
    top = math.ceil(maxy / resolution) * resolution
    width = int(round((right - left) / resolution))
    height = int(round((top - bottom) / resolution))
    transform = from_origin(left, top, resolution, resolution)
    return GridSpec(
        crs=CRS.from_user_input(crs),
        transform=transform,
        width=width,
        height=height,
        resolution=float(resolution),
        bounds=(left, bottom, right, top),
    )


def rasterize_geometry_mask(geometry, grid: GridSpec) -> np.ndarray:
    return features.rasterize(
        [(geometry, 1)],
        out_shape=(grid.height, grid.width),
        transform=grid.transform,
        fill=0,
        default_value=1,
        dtype="uint8",
        all_touched=False,
    )


grid = aligned_grid(processing_geometry, TARGET_CRS, RESOLUTION_M)
aoi_mask = rasterize_geometry_mask(aoi_geometry, grid)
processing_mask = rasterize_geometry_mask(processing_geometry, grid)

n_pixels = grid.width * grid.height
print(f"Grid: {grid.width:,} columns x {grid.height:,} rows = {n_pixels:,} cells")
print(f"Resolution: {grid.resolution:g} m")
print("CRS:", grid.crs)
print("Bounds:", grid.bounds)
print(f"Approximate float32 size per raster: {n_pixels * 4 / 1e6:,.1f} MB")


## 6. Rasterization and neighborhood functions

Distances use an exact Euclidean distance transform after vector rasterization. Neighborhood density uses a circular kernel. Road-length density is a first-pass raster approximation: each road-crossed 30-m cell contributes approximately one cell width of centerline. It is suitable for pilot comparisons but should be sensitivity-tested before final publication.


In [ ]:
def valid_geometries(gdf: gpd.GeoDataFrame | None):
    if gdf is None or gdf.empty:
        return []
    return [geom for geom in gdf.geometry if geom is not None and not geom.is_empty]


def rasterize_binary(
    gdf: gpd.GeoDataFrame | None,
    grid: GridSpec,
    *,
    all_touched: bool = False,
) -> np.ndarray:
    geoms = valid_geometries(gdf)
    if not geoms:
        return np.zeros((grid.height, grid.width), dtype=np.uint8)
    return features.rasterize(
        ((geom, 1) for geom in geoms),
        out_shape=(grid.height, grid.width),
        transform=grid.transform,
        fill=0,
        default_value=1,
        dtype="uint8",
        all_touched=all_touched,
    )


def as_representative_points(gdf: gpd.GeoDataFrame | None) -> gpd.GeoDataFrame | None:
    """Convert any geometry type to one representative point per record."""
    if gdf is None:
        return None
    result = gdf.copy()
    non_points = ~result.geometry.geom_type.isin(["Point", "MultiPoint"])
    if non_points.any():
        result.loc[non_points, "geometry"] = result.loc[non_points, "geometry"].representative_point()
    # MultiPoint records are represented by their centroid for one-record-one-count behavior.
    multi = result.geometry.geom_type.eq("MultiPoint")
    if multi.any():
        result.loc[multi, "geometry"] = result.loc[multi, "geometry"].centroid
    return result


def rasterize_point_values(
    gdf: gpd.GeoDataFrame | None,
    grid: GridSpec,
    *,
    value_column: str | None = None,
) -> np.ndarray:
    points = as_representative_points(gdf)
    if points is None or points.empty:
        return np.zeros((grid.height, grid.width), dtype=np.float32)

    if value_column is None:
        values = np.ones(len(points), dtype=np.float32)
    else:
        values = pd.to_numeric(points[value_column], errors="coerce").fillna(0).to_numpy(np.float32)

    shapes = ((geom, float(value)) for geom, value in zip(points.geometry, values))
    return features.rasterize(
        shapes,
        out_shape=(grid.height, grid.width),
        transform=grid.transform,
        fill=0.0,
        dtype="float32",
        merge_alg=MergeAlg.add,
        all_touched=False,
    )


def distance_to_features(binary_feature_raster: np.ndarray, grid: GridSpec) -> np.ndarray:
    """Distance in meters from every cell to the nearest feature cell."""
    if not np.any(binary_feature_raster):
        return np.full(binary_feature_raster.shape, np.nan, dtype=np.float32)
    distance = distance_transform_edt(
        binary_feature_raster == 0,
        sampling=(grid.resolution, grid.resolution),
    )
    return distance.astype(np.float32)


def circular_kernel(radius_m: float, resolution_m: float) -> np.ndarray:
    radius_cells = max(1, int(math.ceil(radius_m / resolution_m)))
    yy, xx = np.ogrid[-radius_cells : radius_cells + 1, -radius_cells : radius_cells + 1]
    kernel = (xx * xx + yy * yy <= (radius_m / resolution_m) ** 2).astype(np.float32)
    return kernel


def moving_sum(array: np.ndarray, radius_m: float, resolution_m: float) -> tuple[np.ndarray, np.ndarray]:
    kernel = circular_kernel(radius_m, resolution_m)
    result = fftconvolve(array.astype(np.float32), kernel, mode="same").astype(np.float32)
    return result, kernel


def point_density_per_km2(
    point_count_raster: np.ndarray,
    *,
    radius_m: float,
    grid: GridSpec,
) -> np.ndarray:
    counts, kernel = moving_sum(point_count_raster, radius_m, grid.resolution)
    area_km2 = kernel.sum() * grid.resolution**2 / 1_000_000.0
    return np.maximum(counts / area_km2, 0).astype(np.float32)


def approximate_line_density_km_per_km2(
    line_binary_raster: np.ndarray,
    *,
    radius_m: float,
    grid: GridSpec,
) -> np.ndarray:
    # First-order approximation: one road-crossed pixel contributes one pixel width.
    local_length_m = line_binary_raster.astype(np.float32) * grid.resolution
    length_m, kernel = moving_sum(local_length_m, radius_m, grid.resolution)
    area_km2 = kernel.sum() * grid.resolution**2 / 1_000_000.0
    return np.maximum((length_m / 1_000.0) / area_km2, 0).astype(np.float32)


def moving_fraction(
    numerator_binary: np.ndarray,
    valid_mask: np.ndarray,
    *,
    radius_m: float,
    grid: GridSpec,
) -> np.ndarray:
    numerator, _ = moving_sum(numerator_binary.astype(np.float32), radius_m, grid.resolution)
    denominator, _ = moving_sum(valid_mask.astype(np.float32), radius_m, grid.resolution)
    result = np.divide(
        numerator,
        denominator,
        out=np.full_like(numerator, np.nan, dtype=np.float32),
        where=denominator > 0,
    )
    return np.clip(result, 0, 1).astype(np.float32)


def mask_array_to_aoi(
    array: np.ndarray,
    aoi_mask: np.ndarray,
    *,
    nodata: float = NODATA_FLOAT,
) -> np.ndarray:
    result = np.asarray(array, dtype=np.float32).copy()
    result[~np.isfinite(result)] = nodata
    result[aoi_mask == 0] = nodata
    return result


def write_float_raster(
    path: str | Path,
    array: np.ndarray,
    *,
    grid: GridSpec,
    aoi_mask: np.ndarray,
    nodata: float = NODATA_FLOAT,
    overwrite: bool = OVERWRITE,
    tags: Mapping[str, object] | None = None,
) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and not overwrite:
        print("Exists; skipped:", path)
        return path

    output = mask_array_to_aoi(array, aoi_mask, nodata=nodata)
    profile = {
        "driver": "GTiff",
        "height": grid.height,
        "width": grid.width,
        "count": 1,
        "dtype": "float32",
        "crs": grid.crs,
        "transform": grid.transform,
        "nodata": nodata,
        "compress": "deflate",
        "predictor": 3,
        "BIGTIFF": "IF_SAFER",
    }
    with rasterio.open(path, "w", **profile) as dst:
        dst.write(output, 1)
        if tags:
            dst.update_tags(**{key: str(value) for key, value in tags.items()})
    print("Saved:", path)
    return path


def distance_decay(distance_m: np.ndarray, decay_length_m: float) -> np.ndarray:
    """Convert a distance layer to a 0-1 pressure score; decay length is provisional."""
    if decay_length_m <= 0:
        raise ValueError("decay_length_m must be positive")
    result = np.exp(-distance_m / decay_length_m)
    result[~np.isfinite(distance_m)] = np.nan
    return result.astype(np.float32)


## 7. Generate current-reference infrastructure layers

These outputs describe the spatial configuration of the downloaded HOTOSM snapshot. Their filenames intentionally use `reference_snapshot`, not an annual date.

The distance layers are generally the most robust first variables for RSFs. Density layers provide exposure at broader ecological scales.


In [ ]:
REFERENCE_DIR = OUTPUT_ROOT / "reference_snapshot"
REFERENCE_DIR.mkdir(parents=True, exist_ok=True)


def generate_distance_product(
    name: str,
    gdf: gpd.GeoDataFrame | None,
    *,
    all_touched: bool,
    source_note: str,
) -> Path | None:
    if gdf is None or gdf.empty:
        warnings.warn(f"No features available for {name}; output skipped.")
        return None
    feature_raster = rasterize_binary(gdf, grid, all_touched=all_touched)
    distance = distance_to_features(feature_raster, grid)
    return write_float_raster(
        REFERENCE_DIR / f"EM_{name}_distance_m_reference_snapshot.tif",
        distance,
        grid=grid,
        aoi_mask=aoi_mask,
        tags={
            "metric": "euclidean_distance_m",
            "source_temporality": "current_reference_snapshot_not_historical",
            "source_note": source_note,
        },
    )


# Roads by ecological/access class.
for road_group in ["major", "secondary", "local", "track"]:
    subset = roads_std.loc[roads_std["road_group"] == road_group].copy()
    generate_distance_product(
        f"road_{road_group}",
        subset,
        all_touched=True,
        source_note=f"HOTOSM road group: {road_group}",
    )

# All populated places: use both polygons and points when both are available.
place_parts = [places_polygons_std]
if places_points_std is not None:
    place_parts.append(places_points_std)
places_all = gpd.GeoDataFrame(
    pd.concat(place_parts, ignore_index=True),
    geometry="geometry",
    crs=TARGET_CRS,
)
generate_distance_product(
    "populated_place",
    places_all,
    all_touched=True,
    source_note="HOTOSM populated-place points and/or polygons",
)

# Railways retained only where the OSM railway tag appears to identify a rail line.
rail_active = railways_std.loc[railways_std["rail_active_candidate"]].copy()
generate_distance_product(
    "railway",
    rail_active,
    all_touched=True,
    source_note="HOTOSM active-candidate railway classes",
)

# Buildings are optional.
if buildings_std is not None and not buildings_std.empty:
    generate_distance_product(
        "building",
        buildings_std,
        all_touched=True,
        source_note="HOTOSM building polygons; current reference snapshot",
    )


In [ ]:
# Neighborhood density layers.
# All-road and track density are written at selected ecological radii.
all_roads_binary = rasterize_binary(roads_std, grid, all_touched=True)
track_binary = rasterize_binary(
    roads_std.loc[roads_std["road_group"] == "track"],
    grid,
    all_touched=True,
)

for radius_m in DENSITY_RADII_M:
    all_road_density = approximate_line_density_km_per_km2(
        all_roads_binary,
        radius_m=radius_m,
        grid=grid,
    )
    write_float_raster(
        REFERENCE_DIR / f"EM_all_road_density_km_km2_r{radius_m}m_reference_snapshot.tif",
        all_road_density,
        grid=grid,
        aoi_mask=aoi_mask,
        tags={
            "metric": "approximate_line_density_km_per_km2",
            "radius_m": radius_m,
            "method_note": "30-m raster centerline approximation; sensitivity testing required",
            "source_temporality": "current_reference_snapshot_not_historical",
        },
    )

    track_density = approximate_line_density_km_per_km2(
        track_binary,
        radius_m=radius_m,
        grid=grid,
    )
    write_float_raster(
        REFERENCE_DIR / f"EM_track_density_km_km2_r{radius_m}m_reference_snapshot.tif",
        track_density,
        grid=grid,
        aoi_mask=aoi_mask,
        tags={
            "metric": "approximate_line_density_km_per_km2",
            "radius_m": radius_m,
            "method_note": "30-m raster centerline approximation; sensitivity testing required",
            "source_temporality": "current_reference_snapshot_not_historical",
        },
    )

# Settlement density: one count per mapped populated-place record.
place_count_raster = rasterize_point_values(places_all, grid)
for radius_m in [1_000, 5_000]:
    place_density = point_density_per_km2(place_count_raster, radius_m=radius_m, grid=grid)
    write_float_raster(
        REFERENCE_DIR / f"EM_populated_place_density_n_km2_r{radius_m}m_reference_snapshot.tif",
        place_density,
        grid=grid,
        aoi_mask=aoi_mask,
        tags={
            "metric": "mapped_place_records_per_km2",
            "radius_m": radius_m,
            "source_temporality": "current_reference_snapshot_not_historical",
        },
    )

# Building-record density: one count per polygon, using an interior representative point.
if buildings_std is not None and not buildings_std.empty:
    building_count_raster = rasterize_point_values(buildings_std, grid)
    for radius_m in [500, 1_000, 5_000]:
        building_density = point_density_per_km2(
            building_count_raster,
            radius_m=radius_m,
            grid=grid,
        )
        write_float_raster(
            REFERENCE_DIR / f"EM_building_density_n_km2_r{radius_m}m_reference_snapshot.tif",
            building_density,
            grid=grid,
            aoi_mask=aoi_mask,
            tags={
                "metric": "mapped_building_records_per_km2",
                "radius_m": radius_m,
                "source_temporality": "current_reference_snapshot_not_historical",
            },
        )


## 8. Standardize the 2008–2024 field anthropization records

Because the source schema has not yet been inspected here, the mapping is configurable. Set an entry to the exact source field name when automatic matching selects the wrong column.

Unknown records default to `event`, which is conservative: they affect only their observation year rather than being assumed persistent.


In [ ]:
# Set exact source field names here after inspecting anthro_raw.columns.
# Keep None to use automatic candidate matching.
ANTHRO_COLUMN_MAP = {
    "feature_id": None,
    "pressure_type": None,
    "pressure_subtype": None,
    "observation_date": None,
    "observation_year": None,
    "start_year": None,
    "end_year": None,
    "temporal_class": None,
    "intensity": None,
    "confidence": None,
    "survey_effort": None,
}

ANTHRO_CANDIDATES = {
    "feature_id": ["feature_id", "record_id", "station_id", "objectid", "fid", "id"],
    "pressure_type": [
        "pressure_type", "anthropization_type", "anthropisation_type", "category",
        "categorie", "class", "classe", "type",
    ],
    "pressure_subtype": ["pressure_subtype", "subtype", "sub_type", "detail", "description"],
    "observation_date": ["observation_date", "survey_date", "obs_date", "date_obs", "date"],
    "observation_year": ["observation_year", "survey_year", "obs_year", "year", "annee"],
    "start_year": ["start_year", "year_start", "first_year", "debut_annee"],
    "end_year": ["end_year", "year_end", "last_year", "fin_annee"],
    "temporal_class": ["temporal_class", "persistence_class", "persistence", "duration_class"],
    "intensity": ["intensity", "pressure_score", "score", "count", "nombre", "number", "n"],
    "confidence": ["confidence", "confidence_score", "quality", "certainty"],
    "survey_effort": ["survey_effort", "effort", "transect_length", "hours", "observer_days"],
}

# These keyword rules are intentionally editable and should be matched to the
# actual Reneco anthropization legend after inspection.
TEMPORAL_KEYWORDS = {
    "permanent": {
        "building", "buildings", "house", "village", "road", "railway", "well",
        "mine", "quarry", "powerline", "infrastructure", "batiment", "route", "puits",
    },
    "semi_permanent": {
        "track", "enclosure", "permanent_camp", "piste", "enclos",
    },
    "annual": {
        "cultivation", "cultivated", "agriculture", "cropland", "culture",
    },
    "event": {
        "vehicle", "car", "people", "person", "livestock", "grazing", "temporary_camp",
        "tent", "sheep", "goat", "camel", "vehicule", "personne", "betail", "campement",
    },
}


def infer_temporal_class(pressure_type: object) -> str:
    text = normalize_text(pressure_type)
    tokens = set(text.split("_")) | {text}
    for temporal_class, keywords in TEMPORAL_KEYWORDS.items():
        if text in keywords or tokens.intersection(keywords):
            return temporal_class
    return "event"


def normalize_temporal_class(value: object) -> str:
    text = normalize_text(value)
    aliases = {
        "persistent": "permanent",
        "permanent": "permanent",
        "stepwise": "permanent",
        "semi_persistent": "semi_permanent",
        "semi_permanent": "semi_permanent",
        "annual": "annual",
        "yearly": "annual",
        "event": "event",
        "temporary": "event",
        "seasonal": "event",
    }
    return aliases.get(text, text if text in {"permanent", "semi_permanent", "annual", "event"} else "")


def numeric_year(series: pd.Series) -> pd.Series:
    values = pd.to_numeric(series, errors="coerce")
    values = values.where(values.between(1900, 2100))
    return values.round().astype("Int64")


def standardize_anthropization(
    gdf: gpd.GeoDataFrame,
    column_map: Mapping[str, str | None],
) -> gpd.GeoDataFrame:
    result = gdf.copy()
    resolved: dict[str, str | None] = {}
    for standard_name, candidates in ANTHRO_CANDIDATES.items():
        resolved[standard_name] = first_existing_column(
            result,
            candidates,
            explicit=column_map.get(standard_name),
        )

    out = gpd.GeoDataFrame(geometry=result.geometry.copy(), crs=result.crs)

    id_field = resolved["feature_id"]
    out["feature_id"] = (
        result[id_field].astype("string")
        if id_field
        else pd.Series([f"ANTHRO_{i + 1:06d}" for i in range(len(result))], index=result.index)
    )

    type_field = resolved["pressure_type"]
    subtype_field = resolved["pressure_subtype"]
    out["pressure_type"] = result[type_field].astype("string") if type_field else "unknown"
    out["pressure_subtype"] = result[subtype_field].astype("string") if subtype_field else ""
    out["pressure_type_norm"] = out["pressure_type"].map(normalize_text)

    date_field = resolved["observation_date"]
    year_field = resolved["observation_year"]
    start_field = resolved["start_year"]
    end_field = resolved["end_year"]

    out["observation_date"] = (
        pd.to_datetime(result[date_field], errors="coerce") if date_field else pd.NaT
    )
    if year_field:
        out["observation_year"] = numeric_year(result[year_field])
    else:
        out["observation_year"] = out["observation_date"].dt.year.astype("Int64")

    out["start_year"] = numeric_year(result[start_field]) if start_field else out["observation_year"]
    out["end_year"] = numeric_year(result[end_field]) if end_field else pd.Series(pd.NA, index=result.index, dtype="Int64")

    temporal_field = resolved["temporal_class"]
    if temporal_field:
        supplied_temporal = result[temporal_field].map(normalize_temporal_class)
    else:
        supplied_temporal = pd.Series("", index=result.index)
    inferred_temporal = out["pressure_type"].map(infer_temporal_class)
    out["temporal_class"] = supplied_temporal.where(supplied_temporal.ne(""), inferred_temporal)

    intensity_field = resolved["intensity"]
    confidence_field = resolved["confidence"]
    effort_field = resolved["survey_effort"]
    out["intensity"] = (
        pd.to_numeric(result[intensity_field], errors="coerce").fillna(1.0)
        if intensity_field
        else 1.0
    )
    out["confidence"] = (
        pd.to_numeric(result[confidence_field], errors="coerce").fillna(1.0)
        if confidence_field
        else 1.0
    )
    out["survey_effort"] = (
        pd.to_numeric(result[effort_field], errors="coerce")
        if effort_field
        else np.nan
    )
    out["source"] = "field_anthropization_2008_2024"

    if out["observation_year"].isna().all() and out["start_year"].isna().all():
        raise ValueError(
            "No usable observation/start year was found in the field anthropization data. "
            "Update ANTHRO_COLUMN_MAP before generating annual layers."
        )

    print("Resolved anthropization fields:")
    display(pd.Series(resolved, name="source_field").to_frame())
    return out.reset_index(drop=True)


def active_records_for_year(gdf: gpd.GeoDataFrame, year: int) -> gpd.GeoDataFrame:
    temporal = gdf["temporal_class"]
    start = gdf["start_year"].fillna(gdf["observation_year"])
    end = gdf["end_year"]
    observed = gdf["observation_year"]

    persistent = temporal.isin(["permanent", "semi_permanent"])
    persistent_active = persistent & start.notna() & (start <= year) & (end.isna() | (end >= year))

    annual = temporal.eq("annual")
    annual_active = annual & (
        (observed.eq(year))
        | (start.notna() & (start <= year) & end.notna() & (end >= year))
    )

    event_active = temporal.eq("event") & observed.eq(year)
    active_mask = (persistent_active | annual_active | event_active).fillna(False)
    return gdf.loc[active_mask].copy()


def observed_activity_for_year(gdf: gpd.GeoDataFrame, year: int) -> gpd.GeoDataFrame:
    """Only records tied directly to the selected year; useful for activity surfaces."""
    observed_mask = gdf["observation_year"].eq(year).fillna(False)
    return gdf.loc[observed_mask].copy()


In [ ]:
if anthro_raw is not None and not anthro_raw.empty:
    anthro = standardize_anthropization(anthro_raw, ANTHRO_COLUMN_MAP)

    print("Temporal classes:")
    display(anthro["temporal_class"].value_counts(dropna=False).rename_axis("class").to_frame("n"))

    print("Observation years:")
    display(
        anthro["observation_year"]
        .value_counts(dropna=False)
        .sort_index()
        .rename_axis("year")
        .to_frame("n")
    )

    print("Most frequent pressure types:")
    display(anthro["pressure_type_norm"].value_counts(dropna=False).head(30).rename_axis("type").to_frame("n"))

    anthro_summary = (
        anthro.groupby(["observation_year", "temporal_class", "pressure_type_norm"], dropna=False)
        .size()
        .rename("n_records")
        .reset_index()
    )
    anthro_summary.to_csv(OUTPUT_ROOT / "field_anthropization_summary.csv", index=False)
else:
    anthro = None
    print("Field anthropization data are not loaded. Annual activity cells will be skipped.")


## 9. Generate annual pilot layers from field anthropization

For each pilot year, the notebook produces:

- distance to all records considered active in that year;
- density of records observed in that year;
- intensity-weighted activity density observed in that year.

These layers do **not** interpret unsurveyed locations as zero pressure. A survey-effort adjustment should be developed later if effort information is available.


In [ ]:
if anthro is not None:
    annual_records = []

    for year in PILOT_YEARS:
        print("\n" + "=" * 80)
        print("YEAR", year)
        print("=" * 80)

        year_dir = OUTPUT_ROOT / "annual_pilot" / str(year)
        year_dir.mkdir(parents=True, exist_ok=True)

        active = active_records_for_year(anthro, year)
        observed = observed_activity_for_year(anthro, year)

        annual_records.append(
            {
                "year": year,
                "n_active_records": len(active),
                "n_observed_records": len(observed),
                "n_event_records": int(observed["temporal_class"].eq("event").sum()),
                "n_annual_records": int(observed["temporal_class"].eq("annual").sum()),
                "n_persistent_records": int(
                    active["temporal_class"].isin(["permanent", "semi_permanent"]).sum()
                ),
            }
        )

        if not active.empty:
            active_binary = rasterize_binary(active, grid, all_touched=True)
            active_distance = distance_to_features(active_binary, grid)
            write_float_raster(
                year_dir / f"EM_active_anthropization_distance_m_{year}.tif",
                active_distance,
                grid=grid,
                aoi_mask=aoi_mask,
                tags={
                    "year": year,
                    "metric": "distance_to_active_field_anthropization_m",
                    "temporal_rule": "persistent_after_start; annual/event only during supported years",
                },
            )
        else:
            warnings.warn(f"No active anthropization records found for {year}.")

        if not observed.empty:
            observed_counts = rasterize_point_values(observed, grid)
            observed_intensity = rasterize_point_values(observed, grid, value_column="intensity")

            for radius_m in [1_000, 5_000]:
                activity_density = point_density_per_km2(
                    observed_counts,
                    radius_m=radius_m,
                    grid=grid,
                )
                write_float_raster(
                    year_dir / f"EM_observed_activity_density_n_km2_r{radius_m}m_{year}.tif",
                    activity_density,
                    grid=grid,
                    aoi_mask=aoi_mask,
                    tags={
                        "year": year,
                        "metric": "field_records_observed_per_km2",
                        "radius_m": radius_m,
                        "limitation": "not corrected for spatially variable survey effort",
                    },
                )

                intensity_sum, kernel = moving_sum(observed_intensity, radius_m, grid.resolution)
                area_km2 = kernel.sum() * grid.resolution**2 / 1_000_000.0
                intensity_density = np.maximum(intensity_sum / area_km2, 0)
                write_float_raster(
                    year_dir / f"EM_observed_activity_intensity_km2_r{radius_m}m_{year}.tif",
                    intensity_density,
                    grid=grid,
                    aoi_mask=aoi_mask,
                    tags={
                        "year": year,
                        "metric": "sum_of_record_intensity_per_km2",
                        "radius_m": radius_m,
                        "limitation": "intensity definition and survey effort require validation",
                    },
                )
        else:
            warnings.warn(f"No records were observed directly in {year}.")

    annual_record_summary = pd.DataFrame(annual_records)
    display(annual_record_summary)
    annual_record_summary.to_csv(
        OUTPUT_ROOT / "annual_pilot_record_counts.csv",
        index=False,
    )


## 10. Functions for annual habitat-derived land transformation

Run these functions only after the annual habitat maps are ready and their class codes have been confirmed.

The function converts a categorical habitat raster to the fraction of a selected class in each 30-m cell using average resampling, then calculates neighborhood fractions such as cultivated/fallow cover within 1 km. This retains the conceptual difference between habitat state and surrounding land-transformation pressure.


In [ ]:
def class_fraction_on_grid(
    categorical_raster_path: str | Path,
    class_codes: Sequence[int],
    *,
    grid: GridSpec,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Reproject a selected class to the analysis grid as fractional cover.

    Returns
    -------
    fraction : float32 array
        Fraction of valid source pixels assigned to the selected class.
    valid : uint8 array
        Destination cells supported by valid source data.
    """
    path = Path(categorical_raster_path)
    if not path.exists():
        raise FileNotFoundError(path)

    with rasterio.open(path) as src:
        source = src.read(1, masked=True)
        source_values = source.filled(src.nodata if src.nodata is not None else -999999)
        source_valid = (~source.mask).astype(np.float32)
        source_binary = np.isin(source_values, list(class_codes)).astype(np.float32)
        source_binary[source_valid == 0] = NODATA_FLOAT

        destination = np.full((grid.height, grid.width), NODATA_FLOAT, dtype=np.float32)
        valid_destination = np.zeros((grid.height, grid.width), dtype=np.float32)

        reproject(
            source=source_binary,
            destination=destination,
            src_transform=src.transform,
            src_crs=src.crs,
            src_nodata=NODATA_FLOAT,
            dst_transform=grid.transform,
            dst_crs=grid.crs,
            dst_nodata=NODATA_FLOAT,
            resampling=Resampling.average,
        )
        reproject(
            source=source_valid,
            destination=valid_destination,
            src_transform=src.transform,
            src_crs=src.crs,
            src_nodata=0,
            dst_transform=grid.transform,
            dst_crs=grid.crs,
            dst_nodata=0,
            resampling=Resampling.average,
        )

    valid = valid_destination > 0
    destination[~valid] = np.nan
    destination = np.clip(destination, 0, 1)
    return destination.astype(np.float32), valid.astype(np.uint8)


def generate_habitat_transformation_layers(year: int) -> None:
    habitat_path = Path(str(HABITAT_RASTER_TEMPLATE).format(year=year))
    year_dir = OUTPUT_ROOT / "annual_habitat_pressure" / str(year)
    year_dir.mkdir(parents=True, exist_ok=True)

    for class_name, class_codes in HABITAT_CODES.items():
        cell_fraction, valid = class_fraction_on_grid(
            habitat_path,
            class_codes,
            grid=grid,
        )
        write_float_raster(
            year_dir / f"EM_{class_name}_fraction_30m_{year}.tif",
            cell_fraction,
            grid=grid,
            aoi_mask=aoi_mask,
            tags={
                "year": year,
                "source": str(habitat_path),
                "class_codes": class_codes,
                "metric": "fractional_cover_in_30m_cell",
            },
        )

        for radius_m in [500, 1_000, 5_000]:
            neighborhood_fraction = moving_fraction(
                np.nan_to_num(cell_fraction, nan=0.0),
                valid,
                radius_m=radius_m,
                grid=grid,
            )
            write_float_raster(
                year_dir / f"EM_{class_name}_fraction_r{radius_m}m_{year}.tif",
                neighborhood_fraction,
                grid=grid,
                aoi_mask=aoi_mask,
                tags={
                    "year": year,
                    "source": str(habitat_path),
                    "class_codes": class_codes,
                    "metric": "neighborhood_fraction",
                    "radius_m": radius_m,
                },
            )


# Example after annual maps are available:
# for year in FULL_YEAR_RANGE:
#     generate_habitat_transformation_layers(year)


## 11. Quick visual quality control

Use this helper to inspect individual rasters. It masks NoData values and clips the display range at the 2nd and 98th percentiles so isolated extreme values do not dominate the image.


In [ ]:
def plot_raster_quicklook(path: str | Path, title: str | None = None) -> None:
    path = Path(path)
    with rasterio.open(path) as src:
        data = src.read(1).astype(np.float32)
        if src.nodata is not None:
            data[data == src.nodata] = np.nan
        extent = [src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top]

    finite = data[np.isfinite(data)]
    if finite.size == 0:
        raise ValueError(f"Raster has no finite values: {path}")
    vmin, vmax = np.nanpercentile(finite, [2, 98])

    fig, ax = plt.subplots(figsize=(10, 7))
    image = ax.imshow(data, extent=extent, vmin=vmin, vmax=vmax)
    ax.set_title(title or path.stem)
    ax.set_xlabel("Easting (m)")
    ax.set_ylabel("Northing (m)")
    fig.colorbar(image, ax=ax, shrink=0.8)
    fig.tight_layout()
    plt.show()


# Example:
# plot_raster_quicklook(
#     REFERENCE_DIR / "EM_road_major_distance_m_reference_snapshot.tif"
# )


## 12. Save run metadata

This record is important because the downloaded HOTOSM snapshot date, source paths, class rules, grid, and pilot years determine the interpretation of the output.


In [ ]:
run_metadata = {
    "project": "Eastern Morocco anthropogenic pressure",
    "target_crs": TARGET_CRS,
    "resolution_m": RESOLUTION_M,
    "max_context_m": MAX_CONTEXT_M,
    "pilot_years": PILOT_YEARS,
    "full_year_range": [min(FULL_YEAR_RANGE), max(FULL_YEAR_RANGE)],
    "temporal_note": (
        "HOTOSM layers are current reference snapshots and are not automatically "
        "interpreted as annual historical observations."
    ),
    "paths": {
        "aoi": str(AOI_PATH),
        "anthropization": str(ANTHRO_PATH),
        "roads": str(ROADS_PATH),
        "places_polygons": str(PLACES_POLYGONS_PATH),
        "places_points": str(PLACES_POINTS_PATH),
        "railways": str(RAILWAYS_PATH),
        "buildings": str(BUILDINGS_PATH),
        "habitat_template": str(HABITAT_RASTER_TEMPLATE),
        "output_root": str(OUTPUT_ROOT),
    },
    "road_groups": {key: sorted(value) for key, value in ROAD_GROUPS.items()},
    "place_groups": {key: sorted(value) for key, value in PLACE_GROUPS.items()},
    "habitat_codes": HABITAT_CODES,
}

metadata_path = OUTPUT_ROOT / "run_metadata.json"
with metadata_path.open("w", encoding="utf-8") as file:
    json.dump(run_metadata, file, indent=2)
print("Saved:", metadata_path)


## 13. Interpretation and next development stage

Before running all years, inspect:

1. the input inventory and source category counts;
2. the HOTOSM road and populated-place classifications;
3. the field anthropization column mapping and temporal classes;
4. pilot maps for 2008, 2016, and 2024;
5. locations where recent HOTOSM infrastructure is likely being incorrectly interpreted as historical.

The next development stage should add:

- verified start years for infrastructure where historical imagery supports them;
- survey-effort correction or effort-aware validation for field activity;
- annual built-up and cultivated/fallow fractions from the habitat maps;
- fixed scaling thresholds across the complete study period;
- alternative decay distances and neighborhood scales;
- component/domain comparison in spatially blocked RSF validation;
- a separate lagged, non-circular pressure configuration for habitat-change models.

Only after these checks should the component layers be combined into domain indices or a total Human Pressure Index.
